In [ ]:
import numpy as np
import adaptive_latents
from adaptive_latents import datasets
from sim_stim import make_srs, make_slices_tensor
from itertools import cycle
import pathlib
from learn_s_hat_plots import plot_onestep_pred_error_decreasing, make_table
import matplotlib.pyplot as plt

In [ ]:
rng = np.random.default_rng(0)
d = datasets.Zong22Dataset()
data = d.neural_data

srs = make_srs(data, rng, comparison_preset='visualization', n_runs=1, show_tqdm=True)


In [ ]:
%matplotlib inline
# i= 20
# i= 22
i= 40
sr = srs['learning from stim'][0]

fig2, axs2 = plt.subplots(ncols=2, figsize=(10,4), sharex=False, sharey=False, layout='constrained')

# axs2[0].sharex(axs2[1])
# axs2[0].sharey(axs2[1])

latents = sr.log['latents'].slice_by_time(slice(30,None))
axs2[0].plot(latents[:, 0], latents[:, 1], alpha=.1, color='k')
stim_s = sr.log['stim_intended_samples'].t - latents.dt
# axs2[0].plot(latents.slice_by_time(stim_s)[:, 0], latents.slice_by_time(stim_s)[:, 1], '.', color='r', alpha=.2)

l = 1
r = 5.1
ax_n = 0
center_t = sr.log['stim_intended_samples'].t[i]
latents = sr.log['latents'].slice_by_time(slice(center_t-l,center_t+r))
line = axs2[ax_n].plot(latents[:, 0], latents[:, 1])
stim_s = sr.log['stim_intended_samples'].slice_by_time(slice(center_t-l,center_t+r)).t - latents.dt
latents_s = latents.slice_by_time(stim_s).reshape((-1, latents.shape[1]))
axs2[ax_n].plot(latents_s[:, 0], latents_s[:, 1], '.', color='r')

for arrow_index in [17, 50]:
    axs2[0].annotate('',
                       xytext=(latents[arrow_index, 0], latents[arrow_index, 1]),
                       xy=(latents[arrow_index+1, 0], latents[arrow_index+1, 1]),
                       arrowprops=dict(arrowstyle="simple", color='C0'),
                       size=11
                       )


# axs2[2].plot(latents.t, latents);
# for stim_t in stim_s:
#     axs2[2].axvline(stim_t, color='r')



# u = sr.stim_designer.log[i]['pro'].Q[:,0]
u = sr.stim_designer.log[i]['u']
idx = np.argsort(np.abs(u))[::-1]
print(np.linalg.norm(u,ord=0))

high_d = sr.log['high_d_with_stim'].slice_by_time(slice(center_t-l,center_t+r))
axs2[1].plot(high_d.t, high_d[:,idx[:int(np.linalg.norm(u,ord=0))]]);
for stim_t in stim_s:
    axs2[1].axvline(stim_t, color='r')


fig2.savefig(pathlib.Path('/home/jgould/Documents/neurips_2025/generated') / 'zong_stim.svg', bbox_inches="tight")

In [ ]:
stim_s